In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# !pip install happytransformer
# !pip install wordcloud
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
from happytransformer import HappyTextToText, TTSettings, TTTrainArgs
import pickle

2024-10-30 17:11:31.762960: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-30 17:11:31.903260: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-30 17:11:32.631549: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64
2024-10-30 17:

In [2]:
df = pd.read_csv('combined_Chinese.csv')
df = df.rename(columns={'text': 'input', 'correct': 'target'})
df['input'] = df['input'].apply(lambda x: f"grammar: {x}")

In [3]:
print(df)

                                                    input  \
0                  grammar: Also, I have another problem.   
1                             grammar: And life is short.   
2                              grammar: This is try hard!   
3                     grammar: Wish best wishes for her..   
4       grammar: countless beating from their parent i...   
...                                                   ...   
101383  grammar: Yesterday I anesthetized a 86 years-o...   
101384  grammar: I performed an axillary block and gen...   
101385  grammar: The music clip shows a story of a wom...   
101386  grammar: At the department store, I found some...   
101387  grammar: It has already a filter in it and you...   

                                                   target  
0                            I also have another problem.  
1                                          Life is short.  
2                                            to try hard!  
3       I send my best wish

In [4]:
train_ratio = 0.8
train_size = int(train_ratio * len(df))
train_data = df[:train_size]
eval_data = df[train_size:]

train_data.to_csv("train2.csv", index=False)
eval_data.to_csv("eval2.csv", index=False)

In [5]:
happy_tt = HappyTextToText("T5", "t5-base")

args = TTTrainArgs(
    batch_size=4,  # Increase batch size
    learning_rate=5e-5,  # Lower learning rate for more stable convergence
    max_input_length=32,  # Adjust if necessary for longer sentences
    max_output_length=32,  # Adjust for longer output sentences
    num_train_epochs=1,  # Increase the number of training epochs
    eval_ratio=0.1  # Use 10% of the data for evaluation during training
)

happy_tt.train("train2.csv", args=args)

10/30/2024 17:11:36 - INFO - happytransformer.happy_transformer -   Using device: cpu


Generating train split: 0 examples [00:00, ? examples/s]

10/30/2024 17:11:37 - INFO - happytransformer.happy_transformer -   Tokenizing training data...


Map:   0%|          | 0/72999 [00:00<?, ? examples/s]

Map:   0%|          | 0/8111 [00:00<?, ? examples/s]

10/30/2024 17:11:45 - INFO - happytransformer.happy_transformer -   Moving model to cpu
/home/jovyan/.local/lib/python3.8/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss
1,2.327700,2.122361
1825,1.408900,1.249642
3650,1.302600,1.214212
5475,1.293500,1.199154
7300,1.257700,1.184214
9125,1.252300,1.176737
10950,1.242900,1.164242
12775,1.224900,1.151125
14600,1.218800,1.150779
16425,1.232700,1.149265


In [6]:
with open('model2', 'wb') as file:
    pickle.dump(happy_tt, file)

In [7]:
with open('model2', 'rb') as file:
    happy_tt = pickle.load(file)

beam_settings = TTSettings(num_beams=5, min_length=1, max_length=50)
eval2 = pd.read_csv('eval2.csv')
end = 50
# Example list of learner sentences
learner_sentences = eval2.iloc[:end, 0]
target = eval2.iloc[:end, 1]

corrected = []
# Correct each sentence
for sentence in learner_sentences:
    corrected_sentence = happy_tt.generate_text(sentence, args=beam_settings).text
    corrected.append(corrected_sentence)
c = 0
nc = 0
s = 0

for i in range(len(corrected)):
    ls = learner_sentences[i].replace("grammar: ", "")
    print(ls)
    print(corrected[i])
    print(target[i])
    if (corrected[i] == target[i]):
        c = c+1
    else:
        if (corrected[i] == ls):
            nc = nc+1
        else:
            s = s+1

print(c, nc, s)

10/31/2024 00:57:54 - INFO - happytransformer.happy_transformer -   Initializing a pipeline


And what is a perfect teacher-student relationship like?Should there be a formal distance between them?The question has aroused much interest.
What is a perfect teacher-student relationship like? Should there be a formal distance between them? The question has aroused much interest.
And what is a perfect teacher-student relationship like? Should there be a formal distance between them? These questions have aroused much interest. (Strictly speaking there are two questions here but they are closely related such that they seem like the same question said differently. So you may be able to get away with using "this" and "has".)
But their shop was very high price !!
But their shop was very high price !!
However, their shops are very expensive !!
and my son is now 1 years old.
My son is now 1 years old.
My son is now a one year old boy. (Avoid using And, But and Because at the start of your sentences)
How many day that you do not go to work per year?
How many days do you not go to work per y